# Generate scenario-colored availability tables

Generates the LOCA2-Hybrid and WRF data-availability tables used in `about-the-data.qmd`, row-colored by scenario (Historical = grey, SSP2-4.5 = light orange, SSP3-7.0 = orange, SSP5-8.5 = red). Outputs an HTML fragment (for the web build) and a matching PNG (for the PDF build) for each table into `figures/html/` and `figures/static/`.

Edit the data tables below and re-run this notebook to regenerate the outputs — do not hand-edit the generated HTML/PNG files directly.

In [1]:
import colorsys
import textwrap
from pathlib import Path

import matplotlib.pyplot as plt

FIG_DIR = Path("figures")
HTML_DIR = FIG_DIR / "html"
STATIC_DIR = FIG_DIR / "static"
HTML_DIR.mkdir(parents=True, exist_ok=True)
STATIC_DIR.mkdir(parents=True, exist_ok=True)

NAVY = "#1d3557"  # matches table.table > thead > tr > th in styles.css
BORDER = "#595959"  # dark grey cell separators (overrides site default #ccc for these tables)


def hsl_to_hex(h, s, l):
    r, g, b = colorsys.hls_to_rgb(h / 360, l / 100, s / 100)
    return "#{:02x}{:02x}{:02x}".format(round(r * 255), round(g * 255), round(b * 255))


SCENARIO_COLORS = {
    "Historical": hsl_to_hex(0, 0, 92),
    "SSP2-4.5": hsl_to_hex(30, 100, 90),
    "SSP3-7.0": hsl_to_hex(28, 100, 80),
    "SSP5-8.5": hsl_to_hex(0, 85, 87),
}
SCENARIO_COLORS["SSP1-2.6"] = SCENARIO_COLORS["Historical"]  # same grey as Historical
SCENARIO_COLORS

{'Historical': '#ebebeb',
 'SSP2-4.5': '#ffe6cc',
 'SSP3-7.0': '#ffc999',
 'SSP5-8.5': '#fac2c2',
 'SSP1-2.6': '#ebebeb'}

In [2]:
def scenario_color(label):
    return SCENARIO_COLORS[label.replace(" ", "")]


def render_html_table(headers, rows, scenario_col=0):
    """headers: list[str]; rows: list[tuple], row[scenario_col] selects the row color."""
    col_pct = 100 / len(headers)
    colgroup = "\n".join(f'<col style="width: {col_pct:.2f}%;">' for _ in headers)
    th_style = (
        f"background-color: {NAVY}; color: #ffffff; font-weight: bold; "
        f"text-align: center; padding: 10px 16px; border: 1px solid {NAVY};"
    )
    thead = "<tr>" + "".join(f'<th style="{th_style}">{h}</th>' for h in headers) + "</tr>"
    body_rows = []
    for row in rows:
        color = scenario_color(row[scenario_col])
        td_style = (
            f"background-color: {color}; border: 1px solid {BORDER}; "
            f"padding: 10px 20px; text-align: center;"
        )
        cells = "".join(f'<td style="{td_style}">{val}</td>' for val in row)
        body_rows.append(f"<tr>{cells}</tr>")
    tbody = "\n".join(body_rows)
    return (
        f'<table class="table" style="table-layout: fixed; width: 100%; '
        f'border-collapse: collapse; border: 1px solid {BORDER};">\n'
        f"<colgroup>\n{colgroup}\n</colgroup>\n"
        f"<thead>\n{thead}\n</thead>\n<tbody>\n{tbody}\n</tbody>\n</table>\n"
    )


def render_png_table(headers, rows, out_path, scenario_col=0, header_wrap=14):
    wrapped_headers = ["\n".join(textwrap.wrap(h, header_wrap)) or h for h in headers]
    header_lines = max(h.count("\n") + 1 for h in wrapped_headers)

    fig, ax = plt.subplots(figsize=(max(6, len(headers) * 1.7), 0.55 * (len(rows) + header_lines)))
    ax.axis("off")
    cell_text = [[str(v) for v in row] for row in rows]
    cell_colors = [[scenario_color(row[scenario_col])] * len(headers) for row in rows]
    tbl = ax.table(
        cellText=cell_text,
        colLabels=wrapped_headers,
        cellColours=cell_colors,
        colColours=[NAVY] * len(headers),
        cellLoc="center",
        loc="center",
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(11)
    tbl.auto_set_column_width(list(range(len(headers))))
    tbl.scale(1, 1.8)
    for (r, _c), cell in tbl.get_celld().items():
        cell.set_edgecolor(BORDER)
        if r == 0:
            cell.set_text_props(color="white", weight="bold")
            cell.set_height(cell.get_height() * header_lines)
    fig.tight_layout()
    fig.savefig(out_path, dpi=200, bbox_inches="tight")
    plt.close(fig)

In [3]:
# LOCA2-Hybrid data availability by variable group and scenario (tbl-loca2-availability)
loca2_headers = ["Scenario", "Type", "Precipitation & Temperature", "Humidity", "Wind", "Radiation"]
loca2_rows = [
    ("Historical", "GCMs", 15, 13, 11, 14),
    ("Historical", "Ensembles", 70, 51, 46, 51),
    ("SSP2-4.5", "GCMs", 14, 12, 11, 14),
    ("SSP2-4.5", "Ensembles", 33, 25, 24, 30),
    ("SSP3-7.0", "GCMs", 14, 12, 10, 13),
    ("SSP3-7.0", "Ensembles", 62, 45, 38, 45),
    ("SSP5-8.5", "GCMs", 13, 11, 11, 13),
    ("SSP5-8.5", "Ensembles", 34, 25, 26, 29),
]

(HTML_DIR / "tbl-loca2-availability.html").write_text(render_html_table(loca2_headers, loca2_rows))
render_png_table(loca2_headers, loca2_rows, STATIC_DIR / "tbl-loca2-availability.png")

In [4]:
# WRF projection simulations by scenario and spatial resolution (tbl-wrf-projections)
# "1*" cells = additional CESM2 SSP2-4.5/SSP5-8.5 runs at 9 km (see footnote in about-the-data.qmd)
wrf_headers = ["Scenario", "3 km (CA)", "9 km (WECC)", "45 km"]
wrf_rows = [
    ("Historical", "8", "8", "8"),
    ("SSP2-4.5", "0", "1*", "1"),
    ("SSP3-7.0", "8", "8", "8"),
    ("SSP5-8.5", "0", "1*", "1"),
]

(HTML_DIR / "tbl-wrf-projections.html").write_text(render_html_table(wrf_headers, wrf_rows))
render_png_table(wrf_headers, wrf_rows, STATIC_DIR / "tbl-wrf-projections.png")

In [5]:
# Best-estimate year each SSP scenario reaches a given global warming level (tbl-gwl-year-exceedance)
gwl_headers = ["SSP Scenario", "1.5°C", "2.0°C", "2.5°C", "3.0°C"]
gwl_rows = [
    ("SSP 1-2.6", "2033 (2024–2100+)", "–", "–", "–"),
    ("SSP 2-4.5", "2031 (2024–2043)", "2053 (2039–2081)", "2080 (2054–2100+)", "–"),
    ("SSP 3-7.0", "2031 (2024–2041)", "2047 (2037–2061)", "2062 (2049–2080)", "2076 (2060–2097)"),
    ("SSP 5-8.5", "2028 (2022–2037)", "2042 (2034–2054)", "2054 (2044–2069)", "2065 (2053–2083)"),
]

(HTML_DIR / "tbl-gwl-year-exceedance.html").write_text(render_html_table(gwl_headers, gwl_rows))
render_png_table(gwl_headers, gwl_rows, STATIC_DIR / "tbl-gwl-year-exceedance.png")